# Optimizer Step · PyTorch Episode 12
## Data augmentation: same photos, new lessons

Episode 8's CNN put 90.6% of the board exam on the right shelf. Today we test it with changes a person wouldn't even notice: a photo slid 3 pixels, mirrored, or turned 15 degrees. Then one new function, data augmentation, gives it a new version of every practice photo, every time. Then how much is too much, and which change each fix covers. Then Episode 9's unfinished lab: changes you can't see at all, and how a CNN can practise against them.

| Word | Meaning | In shop terms |
|---|---|---|
| data augmentation | a new version of each practice photo, made while studying | the same photos, new lessons |
| slide, mirror, turn | move a photo, swap its left and right, rotate it | the camera moved |
| pad | a black frame around the photo, so a 28 x 28 window can move inside it | room to slide |
| sneaky change | a tiny change to every pixel, chosen to make the CNN wrong | a forged photo |
| sneaky practice | studying with sneaky photos (adversarial training) | training with forgeries |

Every seed is 42 and the GPU runs in deterministic mode. Run the cells in order. Keep `ep12_plots.py`, `cnn_model.pt` and the `data` folder next to this notebook.

## Restart · photos and grading
### Round 1 · Tools, photos and grading

### 1 · Tools
The same tools as Episode 11. `v2` turns a photo for one of today's tests. The last lines check that the photos and Episode 8's CNN are already on this laptop.

In [ ]:
import copy
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets
from torchvision.transforms import v2
import ep12_plots as viz                   # drawing helpers only

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True  # same numbers every run
torch.backends.cudnn.benchmark = False
need = [Path("data/FashionMNIST"), Path("cnn_model.pt")]
for p in need:                             # all on this laptop already
    assert p.exists(), f"missing {p}: see the PDF"
print(torch.__version__, "|", device)

### 2 · The photos
The same 70,000 photos and the same split as Episodes 7 to 11.

In [ ]:
train_set = datasets.FashionMNIST("data", train=True, download=False)
test_set = datasets.FashionMNIST("data", train=False, download=False)
classes = train_set.classes
X = train_set.data.float().unsqueeze(1) / 255   # 1 channel: grey
y = train_set.targets
X_train, y_train = X[:50000], y[:50000]    # practice book
X_val, y_val = X[50000:], y[50000:]        # mock test
X_test = test_set.data.float().unsqueeze(1) / 255   # board exam
y_test = test_set.targets
print(X_train.shape, X_val.shape, X_test.shape)

### 3 · Grading
Episode 10's helpers, 1,000 photos at a time.

In [ ]:
def picks(model, X):                       # the shelf it picks, per photo
    model.eval()                           # exam mode
    out = []
    with torch.no_grad():
        for i in range(0, len(X), 1000):   # 1,000 photos at a time
            scores = model(X[i:i + 1000].to(device))
            out.append(scores.argmax(dim=1).cpu())
    return torch.cat(out)

def accuracy(model, X, y):
    return round((picks(model, X) == y).float().mean().item(), 3)

## Act 1 · Same photos, new lessons
### Round 2 · Break Episode 8's CNN

### 4 · Episode 8's CNN is back
It studied all 50,000 practice photos and scored 90.6% on the board exam. Today we test it in new ways.

In [ ]:
viz.act(1, "Same photos, new lessons",
        "Episode 8's CNN scored 90.6%. Does it really know a shoe?")

def new_cnn():                             # Episode 8's CNN, 20,490 knobs
    return nn.Sequential(
        nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(), nn.Linear(32 * 7 * 7, 10),
    )

cnn = new_cnn().to(device)                 # it studied all 50,000 photos
cnn.load_state_dict(torch.load("cnn_model.pt", map_location=device))
print("Episode 8's CNN · board exam:", accuracy(cnn, X_test, y_test))

### 5 · Three small changes
Slide a photo 3 pixels, mirror it, turn it 15 degrees. A person still sees the same thing. The CNN?

**Chat picks a photo:** 0 to 9999.

In [ ]:
def slide(X, k):                           # k pixels right (minus: left)
    moved = torch.roll(X, shifts=k, dims=3)
    if k > 0:
        moved[..., :k] = 0                 # the columns that came round
    elif k < 0:
        moved[..., k:] = 0
    return moved

def mirror(X):                             # left and right swap
    return torch.flip(X, dims=[3])

def turn(X, degrees):                      # like a tilted camera
    return v2.functional.rotate(X, degrees)

i = 242                                    # chat picks: 0 to 9999
photo = X_val[i:i + 1]
viz.show_changes(cnn, [("as stored", photo),
                       ("3 pixels right", slide(photo, 3)),
                       ("mirrored", mirror(photo)),
                       ("turned 15 degrees", turn(photo, 15))],
                 y_val[i], classes, f"photo {i} · {classes[y_val[i]]}")

### 6 · Four exams for one CNN
The whole mock test, as stored and with each change.

In [ ]:
def four_exams(model, X, y):               # one CNN, four versions
    return {"as stored": accuracy(model, X, y),
            "3 pixels right": accuracy(model, slide(X, 3), y),
            "mirrored": accuracy(model, mirror(X), y),
            "turned 15": accuracy(model, turn(X, 15), y)}

ep8_exams = four_exams(cnn, X_val, y_val)
print(ep8_exams)
viz.scorecard({"Episode 8's CNN": ep8_exams},
              "Mock test: the same 10,000 photos, four ways")

### 7 · Shoes, facing the other way
Sandals, sneakers and ankle boots: as stored, then mirrored.

**Guess first:** how many mirrored shoes does it get right?

### Round 3 · Data augmentation

### 8 · Data augmentation: a new lesson from every photo
Every batch, every photo: mirrored half the time, then slid up to 3 pixels in any direction. Nothing new is saved to disk.

In [ ]:
def augment(xb, pad=3, mirrors=True):      # a new version of every photo
    if mirrors:
        coin = torch.rand(len(xb), device=xb.device) < 0.5
        xb = torch.where(coin[:, None, None, None], mirror(xb), xb)
    big = nn.functional.pad(xb, (pad, pad, pad, pad))  # a black frame
    r, c = torch.randint(0, 2 * pad + 1, (2, len(xb))).tolist()
    return torch.stack([big[k, :, r[k]:r[k] + 28, c[k]:c[k] + 28]
                        for k in range(len(xb))])   # cut 28 x 28 back

torch.manual_seed(42)
viz.show_augmented(X_val[i], augment)

### 9 · One study function
Episode 11's `study()`, with one new argument: `change`, a function applied to every batch before the CNN sees it.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def study(model, name, change=None, X=X_train, y=y_train, epochs=10,
          every=100):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loader = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True)
    curve, seen, done, step, best_acc = [], [], 0, 0, 0
    for epoch in range(epochs):
        for xb, yb in loader:
            model.train()                  # study mode
            xb = xb.to(device)
            if change is not None:
                xb = change(xb)            # today's one new line
            loss = loss_fn(model(xb), yb.to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            done, step = done + len(xb), step + 1
            if step % every == 0:          # a mock test every 100 batches
                curve.append(accuracy(model, X_val, y_val))
                seen.append(done)
                viz.race_chart(name, seen, curve, epochs * len(X))
                if curve[-1] > best_acc:   # new best: snapshot
                    best_acc = curve[-1]
                    best = copy.deepcopy(model.state_dict())
    model.load_state_dict(best)            # back to the best mock test
    return curve

### 10 · The race: plain against augmented
Two fresh CNNs, the same photos, the same recipe. One studies the photos as stored, the other a new version of every photo, every time.

**The poll:** on the normal mock test, who wins?

In [ ]:
viz.new_race()
viz.race_target(accuracy(cnn, X_val, y_val), "Episode 8's CNN")
torch.manual_seed(42)
plain = new_cnn().to(device)
plain_curve = study(plain, "plain")
torch.manual_seed(42)
augmented = new_cnn().to(device)
aug_curve = study(augmented, "augmented", change=augment)
print("best mock test · plain:", max(plain_curve),
      "| augmented:", max(aug_curve))

### 11 · The rematch: four exams, two CNNs
The same four exams as cell 6, then the board exam.

**The poll:** 3 pixels right, the augmented CNN?

### 12 · The board exam
Normal photos, opened once.

In [ ]:
plain_board = accuracy(plain, X_test, y_test)
aug_board = accuracy(augmented, X_test, y_test)
print("board exam · plain:", plain_board, "| augmented:", aug_board)
viz.reveal_board([("plain", plain_board, viz.HOT),
                  ("augmented", aug_board, viz.NEON)],
                 "Board exam · normal photos")

## Act 2 · How much is too much?
### Round 4 · The dial, and each fix on its own

### 13 · How far may a practice photo slide?
The same recipe with a bigger and bigger slide: 0, 3, 8 and 14 pixels, quick versions of 3 epochs each.

**The poll:** more augmentation, a better CNN?

### 14 · Each fix, on its own
Two quick CNNs: one practises only slides, the other only mirrors. Then both changes, for both.

**Guess first:** does practising slides fix mirrored shoes too?

In [ ]:
viz.new_race()
fixes = {}
for name, change in (("slide only", lambda xb: augment(xb, mirrors=False)),
                     ("mirror only", lambda xb: augment(xb, pad=0))):
    torch.manual_seed(42)
    model = new_cnn().to(device)
    study(model, name, change=change, epochs=3)
    fixes[name] = {
        "3 pixels right": accuracy(model, slide(X_val, 3), y_val),
        "shoes, mirrored": accuracy(model, mirror(X_val[shoes]),
                                    y_val[shoes])}
print(fixes)
viz.scorecard(fixes, "Mock test: each fix, on its own")

## Act 3 · The research lab: changes you can't see
### Round 5 · Sneaky changes

### 15 · Sneaky changes
Episode 9's unfinished lab. Nudge every pixel a tiny bit in the direction that makes the CNN **more wrong**. No pixel may move more than the budget.

In [ ]:
viz.act(3, "The research lab",
        "Changes you can't see")

def sneaky(model, X, y, budget, steps=10):
    X, y = X.to(device), y.to(device)
    moved = X.clone()
    for _ in range(steps):
        moved.requires_grad_(True)
        loss = loss_fn(model(moved), y)    # how wrong is it now?
        grad, = torch.autograd.grad(loss, moved)
        with torch.no_grad():
            moved = moved + budget / 4 * grad.sign()   # more wrong
            moved = X + (moved - X).clamp(-budget, budget)
            moved = moved.clamp(0, 1)
    return moved.detach()

def sneaky_accuracy(model, budget):        # budget out of 255
    right = 0
    for k in range(0, 10000, 1000):        # 1,000 photos at a time
        xb, yb = X_test[k:k + 1000], y_test[k:k + 1000]
        moved = sneaky(model, xb, yb, budget / 255)
        right += (picks(model, moved) == yb).sum().item()
    return round(right / 10000, 3)

### 16 · How small a change?
**The poll:** no pixel may move more than 13 out of 255 (5%). Episode 8's CNN on the board exam? And the augmented one?

### 17 · Spot the difference
One photo, before and after a 13/255 change.

In [ ]:
j = 2                                      # one board-exam photo
tricked = sneaky(cnn, X_test[j:j + 1], y_test[j:j + 1], 13 / 255).cpu()
viz.flicker(cnn, X_test[j], tricked, classes, y_test[j])

### Round 6 · The fight back

### 18 · Fight back: practise with sneaky photos
A copy of Episode 8's CNN studies 2 more epochs, and every practice batch is made sneaky first (3 quick steps).

In [ ]:
torch.manual_seed(42)
tough = copy.deepcopy(cnn)                 # starts as Episode 8's CNN
optimizer = torch.optim.Adam(tough.parameters(), lr=0.001)
loader = DataLoader(TensorDataset(X_train, y_train),
                    batch_size=64, shuffle=True)
for epoch in range(1, 3):                  # 2 epochs
    tough.train()
    for xb, yb in loader:
        moved = sneaky(tough, xb, yb, 13 / 255, steps=3)
        loss = loss_fn(tough(moved), yb.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print("epoch", epoch, "done")

### 19 · Did the practice work?
**Guess first:** the sneaky 13/255 test, after practice? And normal photos?

### 20 · The checklist

In [ ]:
viz.checklist(["test the changes the world will make",
               "augment each change you expect, and only those",
               "only changes that keep the label",
               "every batch, never the files",
               "a dial: too much costs normal marks",
               "sneaky changes need sneaky practice"])

## Practice
1. Cell 5: pick one photo from each shelf. Which shelf breaks most when mirrored?
2. Cell 8: change `pad=3` to `pad=1`. How do the 16 lessons look?
3. Cell 10: add `turn` to the augmentation (a random angle between -15 and 15 degrees). Does "turned 15" improve?
4. Cell 14: add a third CNN that practises both changes (the default `augment`), 3 epochs. Does it fix both?
5. Cell 18: practise for 4 epochs instead of 2. What happens to the sneaky score, and to normal photos?